# Day 45：周测 + Dify/Coze 低代码平台（Jupyter Notebook 实战手册）

> **所属阶段**：第 4 阶段 · 第四阶段：Agent 智能体开发与多 Agent 协同 ｜ **第 6 周**：Agent 基础、工具调用与 LangGraph 状态机  
> **GitLab 工单**：`STARMIND-045` ｜ **开发分支**：`feat/day45-weekly-exam-dify-dsl`  
> **配套 HTML5 课件**：`courseware/day45/index.html` ｜ **独立版本快照**：`versions/day45/`

---

## 🎙️ 老陈开场旁白

早啊各位！一转眼第六周就到收官日了。今天分上下两个半场：上半场我们先啃透国内企业落地最常见的 **Dify / Coze 低代码 Agent 平台**——别以为低代码就“不高级”，老陈带你直接拆开 Dify 导出的 **DSL 工作流定义**，自己写一个能解析 `start -> knowledge_retrieval -> if_else -> llm -> end` 节点和 `{{#node.var#}}` 变量引用的本地执行引擎！下半场结合本周核心考点做一次实战周测与“低代码 vs 代码自研”架构选型评估，帮你在老板问“为什么不用 Dify 直接拖一个？”时给出无懈可击的专业答案。

### 承上启下
- **昨日基线**：前六天（Day 39 ~ 44）我们从零手写了 ReAct 引擎、`@tool` + `AgentExecutor`、LangGraph `StateGraph`、SQLite Checkpointer/HITL/并行写作、多 Agent Supervisor 团队以及 MCP Server/Client 与双层记忆。
- **明日预告**：明天（Day 46）我们进入第七周，正式向生产级稳定性开火：治理 Agent 死循环与幻觉参数、搭建类 LangSmith 的 Trace/Span 全链路观测体系、Guardrails 护栏以及语义缓存降本提速。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day45'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-045）

星辰数科运营部门用 Dify 拖拽了一个“客户工单自动分类与初审”工作流，希望研发团队评估能否直接上线，或将其导出的 DSL 迁移到星辰智脑内部统一托管运行。同时团队需要对第六周 Agent 核心知识点进行客观验收。

**今日迭代目标：**
- 深度解析 Dify / Coze 声明式工作流 DSL 结构（节点类型、边连接、`{{#node_id.field#}}` 变量模板与条件分支）。
- 编写纯 Python 的 `DifyDSLRunner` 引擎，支持在本地离线执行类 Dify DSL 工作流并检测环路。
- 构建量化的“低代码 vs 代码自研”技术选型打分矩阵，并完成第六周核心知识点自动评分周测器。

**验收标准：**
- `DifyDSLRunner` 能正确解析变量插值 `{{#start.query#}}`、根据 `if_else` 条件动态选择分支节点并输出完整节点追踪。
- DSL 存在非法环路或孤立节点时，`validate_dsl()` 能够提前拦截报错。
- 选型评估器能根据业务复杂度、私有化部署、定制化并发与交付周期给出客观推荐方案（Dify / Coze / LangGraph），所有单测 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. Dify DSL 的变量引用语法 `{{#node_id.var_name#}}`

在 Dify 导出的 YAML/JSON DSL 中，节点间数据传递不是靠硬编码 Python 变量，而是靠统一模板语法 `{{#node_id.field#}}` 从全局 `VariablePool` 中动态取值。这种设计让画布连线与参数绑定彻底解耦，也是所有低代码工作流引擎的通用范式。

> **⚠️ 老陈避坑**：坑点：在画布上重命名或删除上游节点后，下游 LLM Prompt 里残留的 `{{#old_node.text#}}` 往往变成空字符串，导致线上回复莫名其妙失去上下文。导出 DSL 做静态引用检查能提前规避这类隐蔽 Bug。

### 2. 2. 什么时候该把 Dify 工作流迁移为代码自研？

出现以下三个信号之一时，就该启动向 LangGraph 代码自研的迁移：
1) 画布节点数超过 25 个、连线像蜘蛛网一样没人敢动；
2) 需要对接公司内部复杂的微服务鉴权、分布式锁或数据库事务；
3) 业务要求每次改版必须经过 GitLab MR Code Review 和 50+ 条自动化回归单测才能发布。

> **⚠️ 老陈避坑**：坑点：千万不要把几十行复杂业务逻辑硬塞进 Dify 的“Python 代码执行沙箱节点”里维护——没有 IDE 补全、没有单测、没有版本 diff，三个月后必成无人敢碰的祖传黑盒。

### 3. 3. 第六周高频面试/周测核心考点清单

1) ReAct 为什么必须设置 `Stop Sequence`（如 `Observation:`）？防止模型自问自答伪造工具输出。
2) LangGraph 中 `Checkpointer` 和 `interrupt_before` 如何配合支撑跨小时人工审批？落库挂起释放线程，审批后按 `thread_id` 加载恢复。
3) MCP 的核心价值是什么？用 JSON-RPC 2.0 标准把 M×N 工具适配降维为 M+N。

> **⚠️ 老陈避坑**：坑点：面试被问到“多 Agent 一定比单 Agent 好吗？”时，切忌盲目吹捧多 Agent，必须从延迟、Token 成本、上下文损耗与权限隔离四个维度辩证作答。


## 💻 实操步骤 1：步骤一：实现类 Dify 声明式 DSL 工作流解析与离线执行引擎

- **目标文件**：`versions/day45/src/day45_dify_dsl_runner.py`
- **老陈代码旁白**：我们直接手写一个能跑类 Dify DSL 的本地工作流引擎 `DifyDSLRunner`！它支持 DAG 环路静态检测、`{{#node.var#}}` 变量池插值、`knowledge_retrieval`、`if_else` 条件分支与 `llm`/`end` 节点执行，以后连运营导出的 Dify 流程你都能本地写 pytest 回归测试！


In [ ]:
"""Day 45 - 步骤一：类 Dify 声明式工作流 DSL 解析与离线执行引擎。"""

from __future__ import annotations

import re
from collections import defaultdict, deque
from typing import Any, Dict, List, Tuple

VAR_PATTERN = re.compile(r"\{\{#([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)#\}\}")


class DifyDSLRunner:
    """
    解析并执行类 Dify 声明式工作流 DSL：
    支持节点类型：start | knowledge_retrieval | if_else | llm | end
    """

    def __init__(self, dsl: Dict[str, Any], kb_docs: Dict[str, str] | None = None) -> None:
        self.dsl = dsl
        self.nodes: Dict[str, Dict[str, Any]] = {n["id"]: n for n in dsl.get("nodes", [])}
        self.edges: List[Dict[str, Any]] = dsl.get("edges", [])
        self.kb_docs = kb_docs or {
            "P0": "【P0 紧急工单 SLA】15 分钟内响应，自动抄送值班架构师与运维组。",
            "P2": "【P2 普通工单 SLA】4 个工作小时内响应，优先走知识库自助排查。",
        }

    def validate_dsl(self) -> None:
        """使用 Kahn 拓扑排序检测 DSL 是否存在非法环路或未定义节点引用。"""
        in_degree: Dict[str, int] = {nid: 0 for nid in self.nodes}
        adj: Dict[str, List[str]] = defaultdict(list)

        for edge in self.edges:
            src, dst = edge["source"], edge["target"]
            if src not in self.nodes or dst not in self.nodes:
                raise ValueError(f"DSL 边引用了不存在的节点: {src} -> {dst}")
            adj[src].append(dst)
            in_degree[dst] += 1

        queue = deque([nid for nid, deg in in_degree.items() if deg == 0])
        visited = 0
        while queue:
            cur = queue.popleft()
            visited += 1
            for nxt in adj[cur]:
                in_degree[nxt] -= 1
                if in_degree[nxt] == 0:
                    queue.append(nxt)

        if visited != len(self.nodes):
            raise ValueError("DSL 校验失败：检测到循环依赖（非 DAG 工作流）")

    def _render_template(self, template: str, var_pool: Dict[str, Dict[str, Any]]) -> str:
        """将 {{#node_id.field#}} 替换为变量池中的真实值。"""
        def _replace(match: re.Match[str]) -> str:
            node_id, field_name = match.group(1), match.group(2)
            return str(var_pool.get(node_id, {}).get(field_name, ""))

        return VAR_PATTERN.sub(_replace, template)

    def _find_next_node(self, current_id: str, branch_handle: str = "default") -> str | None:
        for edge in self.edges:
            if edge["source"] == current_id and edge.get("handle", "default") == branch_handle:
                return edge["target"]
        return None

    def run(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        self.validate_dsl()
        start_nodes = [n["id"] for n in self.nodes.values() if n["type"] == "start"]
        if len(start_nodes) != 1:
            raise ValueError("DSL 必须且只能包含一个 start 节点")

        var_pool: Dict[str, Dict[str, Any]] = {}
        trace: List[Dict[str, Any]] = []
        current_id: str | None = start_nodes[0]

        while current_id is not None:
            node = self.nodes[current_id]
            ntype = node["type"]
            branch_handle = "default"

            if ntype == "start":
                var_pool[current_id] = dict(inputs)
            elif ntype == "knowledge_retrieval":
                key_expr = self._render_template(node["query_template"], var_pool)
                matched = self.kb_docs.get(key_expr, "未命中专项 SLA，按常规工单流程处理。")
                var_pool[current_id] = {"context": matched}
            elif ntype == "if_else":
                left = self._render_template(node["left_template"], var_pool)
                right = str(node["equals"])
                passed = left == right
                branch_handle = "true" if passed else "false"
                var_pool[current_id] = {"branch": branch_handle, "matched": passed}
            elif ntype == "llm":
                rendered = self._render_template(node["prompt_template"], var_pool)
                var_pool[current_id] = {"text": f"[{node.get('model', 'qwen-plus')}] {rendered}"}
            elif ntype == "end":
                outputs = {
                    k: self._render_template(v, var_pool)
                    for k, v in node.get("outputs", {}).items()
                }
                var_pool[current_id] = outputs
                trace.append({"node_id": current_id, "type": ntype, "output": outputs})
                return {"outputs": outputs, "trace": trace, "var_pool": var_pool}
            else:
                raise ValueError(f"不支持的 DSL 节点类型: {ntype}")

            trace.append({"node_id": current_id, "type": ntype, "output": var_pool[current_id]})
            current_id = self._find_next_node(current_id, branch_handle)

        raise RuntimeError("DSL 执行意外结束：未到达 end 节点")


def sample_ticket_triage_dsl() -> Dict[str, Any]:
    """星辰数科运营部导出的工单分流演示 DSL。"""
    return {
        "name": "starmind-ticket-triage",
        "nodes": [
            {"id": "start", "type": "start"},
            {"id": "kb", "type": "knowledge_retrieval", "query_template": "{{#start.priority#}}"},
            {"id": "cond", "type": "if_else", "left_template": "{{#start.priority#}}", "equals": "P0"},
            {
                "id": "llm_urgent",
                "type": "llm",
                "model": "qwen-max",
                "prompt_template": "【紧急升级】工单: {{#start.query#}} | 依据: {{#kb.context#}}",
            },
            {
                "id": "llm_normal",
                "type": "llm",
                "model": "qwen-turbo",
                "prompt_template": "【标准处理】工单: {{#start.query#}} | 依据: {{#kb.context#}}",
            },
            {
                "id": "end_urgent",
                "type": "end",
                "outputs": {"reply": "{{#llm_urgent.text#}}", "level": "URGENT"},
            },
            {
                "id": "end_normal",
                "type": "end",
                "outputs": {"reply": "{{#llm_normal.text#}}", "level": "NORMAL"},
            },
        ],
        "edges": [
            {"source": "start", "target": "kb"},
            {"source": "kb", "target": "cond"},
            {"source": "cond", "target": "llm_urgent", "handle": "true"},
            {"source": "cond", "target": "llm_normal", "handle": "false"},
            {"source": "llm_urgent", "target": "end_urgent"},
            {"source": "llm_normal", "target": "end_normal"},
        ],
    }


if __name__ == "__main__":
    runner = DifyDSLRunner(sample_ticket_triage_dsl())
    res = runner.run({"priority": "P0", "query": "核心支付网关 502 超时"})
    print("输出:", res["outputs"])
    print("执行路径:", [t["node_id"] for t in res["trace"]])


### 🔍 步骤 1 代码深度解析

1. `validate_dsl` 用 Kahn 拓扑排序在运行前静态检查环路和断链，比运行到一半卡死安全得多。
2. `_render_template` 精准还原了 Dify 的 `{{#node_id.field#}}` 变量池替换机制。
3. `if_else` 节点通过 `handle='true' / 'false'` 动态决定下游分支，完全对应 Dify 画布上的双出口连线。

**预期运行输出参考：**
```text
输出: {'reply': '[qwen-max] 【紧急升级】工单: 核心支付网关 502 超时 | 依据: 【P0 紧急工单 SLA】15 分钟内响应，自动抄送值班架构师与运维组。', 'level': 'URGENT'}
执行路径: ['start', 'kb', 'cond', 'llm_urgent', 'end_urgent']
```


## 💻 实操步骤 2：步骤二：低代码 vs 代码开发选型评估器与第六周实战周测评分器

- **目标文件**：`versions/day45/src/day45_platform_selector.py`
- **老陈代码旁白**：接下来我们把“低代码 vs 代码开发”的架构选型方法论代码化，并附带一套第六周核心知识点自动评测器，方便团队随时自查掌握程度。


In [ ]:
"""Day 45 - 步骤二：低代码 vs 代码开发量化选型矩阵与第六周自动周测评分器。"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List


@dataclass
class ProjectRequirements:
    private_deployment_required: bool   # 是否必须内网私有化部署
    has_complex_cycles_or_hitl: bool    # 是否有复杂循环状态机或跨小时人工审批
    need_strict_rbac_and_ci: bool       # 是否需要行级鉴权与严格 Git/pytest CI
    delivery_days: int                  # 期望首版交付天数
    maintainer_is_non_dev: bool         # 是否主要由非研发/运营人员维护


def recommend_platform(req: ProjectRequirements) -> Dict[str, object]:
    """根据企业项目约束量化打分，推荐 Coze / Dify / LangGraph 自研方案。"""
    scores: Dict[str, int] = {"Coze": 0, "Dify": 0, "LangGraph": 0}
    reasons: List[str] = []

    if req.private_deployment_required:
        scores["Coze"] -= 10
        scores["Dify"] += 4
        scores["LangGraph"] += 4
        reasons.append("要求私有化部署：排除纯公有云 SaaS 方案")
    else:
        scores["Coze"] += 3

    if req.has_complex_cycles_or_hitl:
        scores["LangGraph"] += 6
        scores["Dify"] += 1
        reasons.append("存在复杂循环状态机/跨小时 HITL：代码编排具备绝对优势")
    else:
        scores["Dify"] += 3
        scores["Coze"] += 2

    if req.need_strict_rbac_and_ci:
        scores["LangGraph"] += 5
        scores["Dify"] += 1
        reasons.append("要求细粒度 RBAC 鉴权与自动化 CI 单测：优先代码自研")

    if req.delivery_days <= 3:
        scores["Coze"] += 4
        scores["Dify"] += 3
        reasons.append("超短周期（<=3天）交付：低代码画布可快速上线 POC")

    if req.maintainer_is_non_dev:
        scores["Dify"] += 4
        scores["Coze"] += 4
        scores["LangGraph"] -= 2
        reasons.append("由业务/运营人员长期维护：低代码可视化平台门槛更低")

    best = max(scores.items(), key=lambda kv: kv[1])[0]
    return {"recommended": best, "scores": scores, "reasons": reasons}


WEEK6_ANSWER_KEY: Dict[str, str] = {
    "Q1_react_stop_seq": "Observation:",
    "Q2_langgraph_persistence": "Checkpointer",
    "Q3_mcp_protocol": "JSON-RPC 2.0",
    "Q4_multi_agent_pattern": "Supervisor",
    "Q5_dify_var_syntax": "{{#node.var#}}",
}


def grade_week6_quiz(submissions: Dict[str, str]) -> Dict[str, object]:
    """第六周核心考点自动评分器（满分 100 分，80 分及以上为通过）。"""
    total = len(WEEK6_ANSWER_KEY)
    correct = 0
    wrong_questions: List[str] = []
    for qid, expected in WEEK6_ANSWER_KEY.items():
        actual = submissions.get(qid, "").strip()
        if actual.lower() == expected.lower():
            correct += 1
        else:
            wrong_questions.append(qid)
    score = round((correct / total) * 100)
    return {
        "score": score,
        "passed": score >= 80,
        "correct_count": correct,
        "wrong_questions": wrong_questions,
    }


if __name__ == "__main__":
    rec = recommend_platform(
        ProjectRequirements(
            private_deployment_required=True,
            has_complex_cycles_or_hitl=True,
            need_strict_rbac_and_ci=True,
            delivery_days=14,
            maintainer_is_non_dev=False,
        )
    )
    print("核心交易系统选型推荐:", rec["recommended"], "| 分数:", rec["scores"])
    quiz = grade_week6_quiz(WEEK6_ANSWER_KEY)
    print("第六周周测得分:", quiz)


### 🔍 步骤 2 代码深度解析

1. `recommend_platform` 把架构师脑海里的隐性经验转化为可解释的权重打分与决策依据列表 `reasons`。
2. `grade_week6_quiz` 覆盖了 ReAct 停止词、Checkpointer、JSON-RPC 2.0、Supervisor 与 Dify 变量语法五大本周必考核心。

**预期运行输出参考：**
```text
核心交易系统选型推荐: LangGraph | 分数: {'Coze': -10, 'Dify': 6, 'LangGraph': 15}
第六周周测得分: {'score': 100, 'passed': True, 'correct_count': 5, 'wrong_questions': []}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day45/tests/test_day45.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 45 自动化单元测试：验证 Dify DSL 引擎分支执行、环路检测与选型矩阵。"""

import pytest
from src.day45_dify_dsl_runner import DifyDSLRunner, sample_ticket_triage_dsl
from src.day45_platform_selector import (
    ProjectRequirements,
    WEEK6_ANSWER_KEY,
    grade_week6_quiz,
    recommend_platform,
)


def test_dify_dsl_runner_branches_correctly():
    runner = DifyDSLRunner(sample_ticket_triage_dsl())

    p0 = runner.run({"priority": "P0", "query": "数据库连接池耗尽"})
    assert p0["outputs"]["level"] == "URGENT"
    assert "15 分钟内响应" in p0["outputs"]["reply"]

    p2 = runner.run({"priority": "P2", "query": "如何申请测试账号"})
    assert p2["outputs"]["level"] == "NORMAL"
    assert "4 个工作小时内响应" in p2["outputs"]["reply"]


def test_dify_dsl_detects_cycle():
    cyclic_dsl = {
        "nodes": [
            {"id": "start", "type": "start"},
            {"id": "a", "type": "llm", "prompt_template": "hi"},
            {"id": "end", "type": "end", "outputs": {}},
        ],
        "edges": [
            {"source": "start", "target": "a"},
            {"source": "a", "target": "start"},
            {"source": "a", "target": "end"},
        ],
    }
    runner = DifyDSLRunner(cyclic_dsl)
    with pytest.raises(ValueError, match="循环依赖"):
        runner.validate_dsl()


def test_platform_selector_and_week6_quiz():
    ops_req = ProjectRequirements(
        private_deployment_required=True,
        has_complex_cycles_or_hitl=False,
        need_strict_rbac_and_ci=False,
        delivery_days=2,
        maintainer_is_non_dev=True,
    )
    assert recommend_platform(ops_req)["recommended"] == "Dify"

    core_req = ProjectRequirements(
        private_deployment_required=True,
        has_complex_cycles_or_hitl=True,
        need_strict_rbac_and_ci=True,
        delivery_days=14,
        maintainer_is_non_dev=False,
    )
    assert recommend_platform(core_req)["recommended"] == "LangGraph"

    full_score = grade_week6_quiz(WEEK6_ANSWER_KEY)
    assert full_score["score"] == 100 and full_score["passed"] is True

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为 Dify DSL 增加悬空变量引用静态检查

**题目要求：**
编写 `find_broken_var_refs(dsl)`，扫描所有节点的 `prompt_template`/`query_template` 中的 `{{#node_id.field#}}`，找出引用了不存在的 `node_id` 的错误配置。

**思路提示：** 用正则 `VAR_PATTERN.findall(text)` 提取所有 `node_id`，与 DSL 中实际定义的节点 ID 集合求差集。


In [ ]:
"""Day 45 作业 1 参考答案：DSL 悬空变量引用静态检查器。"""

import re
from typing import Any, Dict, List

VAR_RE = re.compile(r"\{\{#([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)#\}\}")


def find_broken_var_refs(dsl: Dict[str, Any]) -> List[str]:
    valid_ids = {n["id"] for n in dsl.get("nodes", [])}
    broken: List[str] = []
    for node in dsl.get("nodes", []):
        for key in ("prompt_template", "query_template", "left_template"):
            text = node.get(key, "")
            for ref_node, ref_field in VAR_RE.findall(text):
                if ref_node not in valid_ids:
                    broken.append(f"{node['id']}->{ref_node}.{ref_field}")
    return broken


if __name__ == "__main__":
    bad_dsl = {
        "nodes": [
            {"id": "start", "type": "start"},
            {"id": "llm1", "type": "llm", "prompt_template": "你好 {{#deleted_node.text#}}"},
        ]
    }
    issues = find_broken_var_refs(bad_dsl)
    print("发现悬空变量引用:", issues)
    assert issues == ["llm1->deleted_node.text"]


**答案解析与评分标准：**

静态分析 DSL 中的悬空变量引用，可以在发布前 100% 拦截画布删节点导致的线上空上下文事故。


### 作业 2（企业实战）：将 Dify DSL 一键转换为 Mermaid 流程图

**题目要求：**
编写 `dsl_to_mermaid(dsl)`，遍历 DSL 的 `nodes` 与 `edges`，输出标准 `graph LR` Mermaid 源码，方便自动嵌入 GitLab MR 文档。

**思路提示：** 对每个节点输出 `id[id: type]`，对每条边输出 `source -->|handle| target`。


In [ ]:
"""Day 45 作业 2 参考答案：Dify DSL 自动转 Mermaid 流程图。"""

from typing import Any, Dict, List


def dsl_to_mermaid(dsl: Dict[str, Any]) -> str:
    lines: List[str] = ["graph LR"]
    for n in dsl.get("nodes", []):
        lines.append(f'    {n["id"]}["{n["id"]} ({n["type"]})"]')
    for e in dsl.get("edges", []):
        handle = e.get("handle")
        if handle:
            lines.append(f'    {e["source"]} -->|{handle}| {e["target"]}')
        else:
            lines.append(f'    {e["source"]} --> {e["target"]}')
    return "\n".join(lines)


if __name__ == "__main__":
    demo = {
        "nodes": [{"id": "start", "type": "start"}, {"id": "end", "type": "end"}],
        "edges": [{"source": "start", "target": "end"}],
    }
    md = dsl_to_mermaid(demo)
    print(md)
    assert "start --> end" in md


**答案解析与评分标准：**

把低代码 DSL 纳入 Git 仓库并在 MR 中自动生成 Mermaid 拓扑图，能让低代码资产也具备工程级的可评审性。


### 作业 3（架构进阶）：设计低代码向代码自研迁移的渐进式灰度策略

**题目要求：**
编写一个 `TrafficSplitter` 类，支持按百分比 `langgraph_ratio`（0.0~1.0）与 `user_id` 哈希稳定分流，将部分流量从 Dify 平滑切流至 LangGraph 自研服务，保证同一用户始终命中同一后端。

**思路提示：** 使用 `hashlib.md5(user_id.encode()).hexdigest()` 取前 8 位转整数对 100 取模，与 `int(ratio * 100)` 比较。


In [ ]:
"""Day 45 作业 3 参考答案：基于一致性哈希的低代码->自研平滑灰度切流器。"""

import hashlib


class TrafficSplitter:
    def __init__(self, langgraph_ratio: float = 0.2) -> None:
        self.threshold = int(max(0.0, min(1.0, langgraph_ratio)) * 100)

    def route(self, user_id: str) -> str:
        bucket = int(hashlib.md5(user_id.encode("utf-8")).hexdigest()[:8], 16) % 100
        return "langgraph" if bucket < self.threshold else "dify"


if __name__ == "__main__":
    splitter = TrafficSplitter(langgraph_ratio=0.3)
    first = splitter.route("emp_10086")
    second = splitter.route("emp_10086")
    assert first == second
    print("用户 emp_10086 稳定命中:", first)


**答案解析与评分标准：**

使用确定性哈希分桶灰度切流，能避免同一个用户在多轮对话中一会儿落到 Dify、一会儿落到 LangGraph 导致会话状态撕裂。


---

## 🎯 老陈收尾复盘

第六周圆满收官！把 Dify DSL 亲手解析一遍后，你会发现低代码平台底层其实就是“可视化画布 + JSON/YAML 声明式 DAG + 沙箱节点执行器”，和我们学的 LangGraph 殊途同归。记住老陈的选型铁律：POC 验证和运营类轻量问答大胆上 Dify/Coze 抢时间，核心交易链路、深度权限控制和复杂状态机果断上代码自研！周末好好休息，下周我们攻克 Agent 工程化护栏和阶段项目三！
